Cambios: 
- Implementacion noiseless simple
- Simulación via qiskit_aer (en GPUs)
- Optimicación pytorch para evaluacion en multiples GPUs

## Implementation (statevector simulation)

In [27]:
#--- INSTALATION INSTRUCTIONS ---#

# For linux 64-bit systems,
#uname -a

# Conda quick installation
#mkdir -p ~/miniconda3
#wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O ~/miniconda3/miniconda.sh
#bash ~/miniconda3/miniconda.sh -b -u -p ~/miniconda3
#rm ~/miniconda3/miniconda.sh

# Create enviroment with conda
#conda create -n myenv python=3.10
#conda activate myenv
#pip install qiskit qiskit-machine-learning 'qiskit-machine-learning[sparse]' qiskit_aer qiskit_algorithms torch matplotlib pylatexenc ipykernelc
# IMPORTANT: Make sure you are on 3.10
# May need to restart the kernel after instalation

#--- Imports ---#
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import random_statevector, Statevector, SparsePauliOp
from qiskit.circuit.library import real_amplitudes, efficient_su2
from qiskit import qpy

from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.gradients import ParamShiftEstimatorGradient, SPSAEstimatorGradient
from qiskit_machine_learning.connectors import TorchConnector

from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as EstimatorV2_sim
from qiskit_aer.quantum_info import AerStatevector

from qiskit_algorithms.gradients import ReverseEstimatorGradient

import numpy as np
import torch
import matplotlib.pyplot as plt
import time
import os
import signal
import datetime as dt

In [28]:
#- Configuration -#

# Training configuration dict
train_config = {
    'execution_type': "noiseless_simulation", #noiseless_simulation
    'n_qubits': 8,
    'seed': 2,
    'id': None, # For different circuits or training parameters
    'reset_data': True,

    'create_circuits': True, # Create circuits manually or load from file
    'gradient_method': "REG", # qiskit_algorithms.gradients For now: PSR, SPSA and REG
    'max_iterations': 1000,
    'gen_iterations': 1,
    'disc_iterations': 1,
    'save_loss_iterations': 10, # Calculate extra forward pass to save loss
    'print_progress_iterations': 10,

    'TorchConnector': True, # Use TorchConnector QNN instances

    'training_data_file': None, # Automatically created with manage_files function
    'circuits_file': None, # Automatically created with manage_files function
    'backend_file': None # Automatically created with manage_files function
}


# File management
def manage_files(
        data_folder_name = 'data', 
        implementation_name = 'noiseless_torch_opt', 
        execution_type_name = train_config['execution_type'], 
        training_data_file_name = 'training_data', 
        circuits_file_name = 'circuits', 
        backend_file_name = 'backend'
        ):
    
    data_folder = data_folder_name + '/' + implementation_name + '/' + execution_type_name + '/' + 'q' + str(train_config['n_qubits']) + '/' + 'seed' + str(train_config['seed']) + '/'
    if train_config['id'] is not None:
        data_folder = data_folder + '/' + str(train_config['id']) + '/' 
    training_data_file = data_folder + training_data_file_name + '.pth'
    circuits_file = data_folder + circuits_file_name + '.qpy'
    backend_file = data_folder + backend_file_name + '.pkl'

    # Create folders if they do not exist
    if not os.path.exists(data_folder):
        os.makedirs(data_folder)

    return training_data_file, circuits_file, backend_file



if ((train_config['training_data_file'] is None) and (train_config['circuits_file'] is None) and (train_config['backend_file'] is None)):
    train_config['training_data_file'], train_config['circuits_file'], train_config['backend_file'] = manage_files()

In [29]:
#- Backend configuration -#
backend_config = {
    # Real backend
    'name': "ibm_basquecountry",
    'channel': "ibm_quantum_platform",

    # Noisy backend
    'reset_backend': False, # Get current backend state or load from file
    'timestamp': dt.datetime.now(), # dt.datetime(year=2026, month=12, day=5, hour = 10, tzinfo=dt.timezone.utc), # Get exact backend state, None to get current state (no he conseguido q funcione) TODO?

    # Noiseless backend
    'sim_options': {
        'method': 'statevector', # automatic, stabilizer (for clifford (simple) gates), matrix_product_state (low entanglement, more qubits), density_matrix (noise simulation, more memory (2^{2n} vs 2^n))
        #'device': 'GPU', # Para nvidia cuda
        'precision': 'single',       # Significant speedup 
        #'cuStateVec_enable': True,   # NVIDIA library optimization
        #'batched_shots_gpu': True,   # Parallelize batch on GPU [9]
        #'blocking_enable': False,     # Disable chunking; simulation fits in VRAM 
        #'target_gpus':[0,1],
        #'seed_simulator': train_config['seed']
    }
}

backend = AerSimulator(**backend_config['sim_options'])

precision = 0.0
estimator = EstimatorV2_sim(
    options = {
        "default_precision": precision,
        #'seed_estimator': train_config['seed'],
        "backend_options": backend_config['sim_options'],
        "run_options": {}
    })

#pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
pm = None


# Select device torch
if torch.cuda.is_available():
    print(f"GPUs available to PyTorch: {torch.cuda.device_count()}")
    os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

In [30]:
#- Create quantum circuits -#

# Create real data sample circuit
def generate_real_circuit():
    n_qubits = train_config['n_qubits']

    # sv = random_statevector(2**N_QUBITS, seed=SEED)
    # qc = QuantumCircuit(N_QUBITS)
    # qc.prepare_state(sv, qc.qubits, normalize=True)

    qc = QuantumCircuit(n_qubits)
    qc.h(range(n_qubits-1))
    qc.cx(n_qubits-2, n_qubits-1)
    return qc


# Create generator
def generate_generator():
    n_qubits = train_config['n_qubits']

    qc = real_amplitudes(n_qubits,
                        reps=3, # Number of layers
                        parameter_prefix='θ_g',
                        name='Generator')
    
    return qc.decompose()


# Create discriminator
def generate_discriminator():
    n_qubits = train_config['n_qubits']

    qc = efficient_su2(n_qubits,
                      entanglement="reverse_linear",
                      reps=1, # Number of layers
                      parameter_prefix='θ_d',
                      name='Discriminator').decompose()


    param_index = qc.num_parameters

    for i in reversed(range(n_qubits - 1)):
        qc.cx(i, n_qubits - 1)

    #qc.rx(disc_weights[param_index], N_QUBITS-1); param_index += 1
    qc.ry(Parameter("θ_d["+str(param_index)+"]"), n_qubits-1); param_index += 1
    qc.rz(Parameter("θ_d["+str(param_index)+"]"), n_qubits-1); param_index += 1
    
    return qc


# Create quantum circuits
def create_circuits():
    real_circuit = generate_real_circuit()
    generator_circuit = generate_generator()
    discriminator_circuit = generate_discriminator()

    # Save circuits in a file
    with open(train_config['circuits_file'], 'wb') as fd:
        qpy.dump([real_circuit, generator_circuit, discriminator_circuit], fd)


# Rewrite circuits file if indicated in options
if train_config['create_circuits']:
    create_circuits()


# Load circuits from file
try:
    with open(train_config['circuits_file'], 'rb') as fd:
        circuits = qpy.load(fd)
except FileNotFoundError:
    print("Circuits file not found. Creating new circuits file.")
    create_circuits()
    with open(train_config['circuits_file'], 'rb') as fd:
        circuits = qpy.load(fd)
    
    
real_circuit = circuits[0]
generator_circuit = circuits[1]
discriminator_circuit = circuits[2]

In [31]:
#- Set up training quantum circuits -#
def generate_training_circuits(real_circuit, generator_circuit, discriminator_circuit):
    n_qubits = train_config['n_qubits']

    # Connect real data and discriminator
    real_disc_circuit = QuantumCircuit(n_qubits)
    real_disc_circuit.compose(real_circuit, inplace=True)
    real_disc_circuit.compose(discriminator_circuit, inplace=True)

    # Connect generator and discriminator
    gen_disc_circuit = QuantumCircuit(n_qubits)
    gen_disc_circuit.compose(generator_circuit, inplace=True)
    gen_disc_circuit.compose(discriminator_circuit, inplace=True)


    # Gradient computation method
    if train_config['gradient_method'] == 'SPSA':
        gradient = SPSAEstimatorGradient(estimator=estimator)
    elif train_config['gradient_method'] == 'REG':
        gradient = ReverseEstimatorGradient()
    else:
        gradient = ParamShiftEstimatorGradient(estimator=estimator)


    # Observables
    H1 = SparsePauliOp.from_list([("Z" + "I"*(n_qubits-1), 1.0)])


    N_DPARAMS = discriminator_circuit.num_parameters

    # specify QNN to update generator parameters
    gen_qnn = EstimatorQNN(circuit=gen_disc_circuit,
                        input_params=gen_disc_circuit.parameters[:N_DPARAMS], # fixed parameters (discriminator parameters)
                        weight_params=gen_disc_circuit.parameters[N_DPARAMS:], # parameters to update (generator parameters)
                        estimator=estimator,
                        observables=[H1],
                        gradient=gradient,
                        default_precision=precision,
                        pass_manager=pm,
                        input_gradients=train_config['TorchConnector']
                        )

    # specify QNN to update discriminator parameters regarding to fake data
    disc_fake_qnn = EstimatorQNN(circuit=gen_disc_circuit,
                            input_params=gen_disc_circuit.parameters[N_DPARAMS:], # fixed parameters (generator parameters)
                            weight_params=gen_disc_circuit.parameters[:N_DPARAMS], # parameters to update (discriminator parameters)
                            estimator=estimator,
                            observables=[H1],
                            gradient=gradient,
                            default_precision=precision,
                            pass_manager=pm,
                            input_gradients=train_config['TorchConnector']
                            )

    # specify QNN to update discriminator parameters regarding to real data
    disc_real_qnn = EstimatorQNN(circuit=real_disc_circuit,
                            input_params=[], # no input parameters
                            weight_params=real_disc_circuit.parameters[:N_DPARAMS], # parameters to update (discriminator parameters)
                            estimator=estimator,
                            observables=[H1],
                            gradient=gradient,
                            default_precision=precision,
                            pass_manager=pm,
                            input_gradients=train_config['TorchConnector']
                            )
    

    return gen_qnn, disc_fake_qnn, disc_real_qnn

gen_qnn, disc_fake_qnn, disc_real_qnn = generate_training_circuits(real_circuit, generator_circuit, discriminator_circuit)

In [32]:
if train_config['TorchConnector']: 
    #f_loss = torch.nn.MSELoss(reduction="sum")
    class FLoss(torch.nn.Module):
        def __init__(self, reduction='sum'):
            super(FLoss, self).__init__()
            self.reduction = reduction

        def forward(self, x, label):
            target_sign = label.to(x.dtype) * 2 - 1
            
            loss = x * target_sign
            
            if self.reduction == 'mean': # Para batches
                return loss.mean()
            elif self.reduction == 'sum':
                return loss.sum()
            return loss
        
    f_loss = FLoss()

In [ ]:
#- Restore parameters and model states -#

# Reset all data training
def reset_data(n_gen_params, n_disc_params):
    np.random.seed(train_config['seed'])

    init_gen_params = np.random.uniform(low=-np.pi, high=np.pi, size=(n_gen_params,)) * 0.1 # Start from near 0 parameters to mitigate drastic changes at the start
    init_disc_params = np.random.uniform(low=-np.pi, high=np.pi, size=(n_disc_params,)) * 0.1

    gen_params = torch.tensor(init_gen_params, requires_grad=True, dtype = torch.float32)
    disc_params = torch.tensor(init_disc_params, requires_grad=True, dtype = torch.float32)

    optimizer_g = torch.optim.Adam([gen_params], lr=0.005)
    optimizer_d = torch.optim.Adam([disc_params], lr=0.005)

    params = {
        'init_gen_params': init_gen_params,
        'init_disc_params': init_disc_params,
        'gen_params': gen_params,
        'disc_params': disc_params,
        'best_gen_params': init_gen_params,
        'optimizer_g_state': optimizer_g.state_dict(),
        'optimizer_d_state': optimizer_d.state_dict(),
        'current_epoch': 0,
        "metrics": {
            "gloss": {},
            "dloss": {},
            "eval": {},
            'times': {},
        },
        'random_state': np.random.get_state()
    }

    if train_config['TorchConnector']:
        model_g = TorchConnector(gen_qnn, gen_params)
        model_dr = TorchConnector(disc_real_qnn, disc_params)
        model_df = TorchConnector(disc_fake_qnn)

        # Force model2 to look at model1's weights
        if 'weight' in model_df._parameters:
            model_df._parameters.pop('weight')
        if '_weights' in model_df._parameters:
            model_df._parameters.pop('_weights')
        model_df._parameters['weight'] = model_dr.weight
        model_df._parameters['_weights'] = model_dr.weight

        model_g.train() 
        model_dr.train()
        model_df.train()

        params['model_g_state'] = model_g.state_dict(),
        params['model_dr_state'] = model_dr.state_dict(),
        params['model_df_state'] = model_df.state_dict(),

    torch.save(params, train_config['training_data_file'])


# Load parameters and training states
if train_config['reset_data']:
    reset_data(generator_circuit.num_parameters, discriminator_circuit.num_parameters)

try:
    params = torch.load(train_config['training_data_file'], weights_only=False)
except FileNotFoundError:
    print("Training data file not found. Resetting parameters.")
    reset_data(generator_circuit.num_parameters, discriminator_circuit.num_parameters)
    params = torch.load(train_config['training_data_file'], weights_only=False)

np.random.set_state(params['random_state'])

gen_params = params['gen_params']
disc_params = params['disc_params']

optimizer_g = torch.optim.Adam([gen_params])
optimizer_d = torch.optim.Adam([disc_params])

optimizer_g.load_state_dict(params['optimizer_g_state'])
optimizer_d.load_state_dict(params['optimizer_d_state'])

current_epoch = params['current_epoch']
gloss = params['metrics']['gloss']
gen_loss = list(gloss)[-1] if (gloss) else None
dloss = params['metrics']['dloss']
disc_loss = list(dloss)[-1] if (dloss) else None
eval = params['metrics']['eval']
min_eval = np.min(list(eval.values())) if (eval) else float('inf')
best_gen_params = params['best_gen_params']
times = params['metrics']['times']


if train_config['TorchConnector']:
    model_g = TorchConnector(gen_qnn)    
    model_dr = TorchConnector(disc_real_qnn)
    model_df = TorchConnector(disc_fake_qnn)
    model_g.load_state_dict(params['model_g_state'])
    model_dr.load_state_dict(params['model_dr_state'])
    model_df.load_state_dict(params['model_df_state'])

    # Force model2 to look at model1's weights
    if 'weight' in model_df._parameters:
        model_df._parameters.pop('weight')
    if '_weights' in model_df._parameters:
        model_df._parameters.pop('_weights')
    model_df._parameters['weight'] = model_dr.weight
    model_df._parameters['_weights'] = model_dr.weight

    optimizer_g = torch.optim.Adam(model_g.parameters())
    optimizer_d = torch.optim.Adam(model_dr.parameters())
    optimizer_g.load_state_dict(params['optimizer_g_state'])
    optimizer_d.load_state_dict(params['optimizer_d_state'])

    # --- VERIFICATION ---
    print(f"Are weights tied? {model_df.weight is model_dr.weight}") # Should be True

In [24]:
#- Manage training interruption -#

# Class to manage training interruption
class Interrupter:
    def __init__(self):
        self.kill_now = False
        self.interrupt_count = 0

        # Intercept the Ctrl+C signal
        signal.signal(signal.SIGINT, self.handle_signal)
        # Intercept the termination signal (useful for Docker/systems)
        #signal.signal(signal.SIGTERM, self.handle_signal)

    def handle_signal(self, signum, frame):
        self.interrupt_count += 1
        
        if self.interrupt_count == 1:
            # First Press: Enable graceful exit
            self.kill_now = True
            print("\nInterrupter: Termination signal received. The loop will stop after the current iteration. (Press Ctrl+C again to force quit)")
        
        elif self.interrupt_count >= 2:
            # Second Press: Force quit immediately
            print("\nInterrupter: [!] Force quit triggered! Terminating immediately.")
            # Restore default signal handler to avoid recursion
            signal.signal(signal.SIGINT, signal.SIG_DFL)
            # Raise the exception to stop execution right here
            raise KeyboardInterrupt

In [ ]:
#- Evualuation method -#

# Evaluation method: KL-Div of generated (ger_dist) and real (target) sample
def evaluate(gen_dist, target):
    return torch.nn.functional.kl_div(
        input = gen_dist.log(),
        target = target, 
        reduction = 'sum' 
    ).item()

In [ ]:
#- Forward and backward pass -#

if train_config['TorchConnector']:
    # Discriminator pass
    def disc_pass():
        optimizer_d.zero_grad()

        # Calculate discriminator gradient with real data
        real_output = model_dr()
        real_loss = f_loss(real_output, torch.ones(1)) # 1-> Real guess (correct) # TODO device=device) OPTIMIZA
        real_loss.backward(retain_graph=True) # =False for V2: two parameter updates
        
        #optimizer_d.step() # V2: two parameter updates

        # Calculate discriminator gradient with generated data
        #optimizer_d.zero_grad() # V2: two parameter updates
        gen_params = torch.nn.utils.parameters_to_vector(model_g.parameters()).detach() #gen_params = optimizer_g.param_groups[0]['params'][0].detach()
        fake_output = model_df(gen_params)
        fake_loss = f_loss(fake_output, torch.zeros(1)) # 0-> Fake guess (correct) # TODO device=device)
        fake_loss.backward()

        optimizer_d.step()

        # Calculate discriminator cost
        disc_loss = (real_loss + fake_loss -2)/4
        if (disc_train_step == D_STEPS-1):
            dloss[epoch] = disc_loss.detach().numpy()

    # Generator pass
    def gen_pass():
        # Calculate generator gradient and update parameters
        optimizer_g.zero_grad()  # Initialize/clear gradients
        disc_params = torch.nn.utils.parameters_to_vector(model_dr.parameters()).detach() #disc_params = optimizer_d.param_groups[0]['params'][0].detach()
        gen_output = model_g(disc_params)
        gen_loss = f_loss(gen_output, torch.ones(1)) # 1-> Real guess (decieved) # TODO device=device)
        gen_loss.backward()  # Backward pass
        optimizer_g.step()

        # Save generator cost
        gen_loss = (gen_loss.detach().numpy() -1)/2
        if (gen_train_step == G_STEPS-1):
            gloss[epoch] = gen_loss

    # Evaluate performance
    def evaluation():

else:
    # Discriminator pass
    def disc_pass():
        gen_params_np = gen_params.detach().cpu().numpy() # TODO no es necesario, con una vez ya vale, memoria compartida no?
        disc_params_np = disc_params.detach().cpu().numpy()

        # Calculate discriminator cost
        if (disc_train_step == D_STEPS - 1) and (epoch % C_STEPS == 0):
            value_dcost_fake = disc_fake_qnn.forward(gen_params_np, disc_params_np)[0,0]
            value_dcost_real = disc_real_qnn.forward([], disc_params_np)[0,0]
            disc_loss = ((value_dcost_real - value_dcost_fake)-2)/4
            dloss[epoch] = disc_loss

        # Caltulate discriminator gradient
        grad_dcost_fake = disc_fake_qnn.backward(gen_params_np, disc_params_np)[1][0,0]
        grad_dcost_real = disc_real_qnn.backward([], disc_params_np)[1][0,0]
        grad_dcost = grad_dcost_real - grad_dcost_fake
        
        # Update discriminator parameters
        optimizer_d.zero_grad()
        disc_params.grad = torch.tensor(grad_dcost, dtype = torch.float32, device=device)
        optimizer_d.step()

    # Generator pass
    def gen_pass():
        gen_params_np = gen_params.detach().cpu().numpy()
        disc_params_np = disc_params.detach().cpu().numpy()

        # Calculate generator cost
        if (gen_train_step == G_STEPS - 1) and (epoch % C_STEPS == 0):
            value_gcost = gen_qnn.forward(disc_params_np, gen_params_np)[0,0]
            gen_loss = (value_gcost-1)/2
            gloss[epoch] = gen_loss

        # Calculate generator gradient
        grad_gcost = gen_qnn.backward(disc_params_np, gen_params_np)[1][0,0]

        # Update generator parameters
        optimizer_g.zero_grad()
        gen_params.grad = torch.tensor(grad_gcost, dtype = torch.float32, device=device)
        optimizer_g.step()

    # Evaluate performance
    def evaluation():
        sv = AerStatevector(generator_circuit.assign_parameters(gen_params_np), device=device)
        gen_dist = torch.tensor(sv.probabilities(), device=device)

        # Performance measurement function: uses Kullback Leibler Divergence to measures the distance between two distributions
        current_kl = evaluate(gen_dist, real_distribution_tensor)
        
        eval[epoch] = current_kl
        if min_eval > current_kl:
            min_eval = current_kl
            best_gen_params = gen_params_np.copy() # New best


In [ ]:
# #- Training -#

# D_STEPS = train_config['disc_iterations']
# G_STEPS = train_config['gen_iterations']
# C_STEPS = train_config['save_loss_iterations']

# real_distribution_tensor = torch.from_numpy(Statevector(real_circuit).probabilities()).to(device) # Retrieve real data probability distribution 

# interrupter = Interrupter()

# if train_config['print_progress_iterations']:
#     TABLE_HEADERS = "Epoch | Generator cost | Discriminator cost | Eval | Best eval | Time |"
#     print(TABLE_HEADERS)

# prev_times = 0
# start_time = time.time()

# #--- Training loop ---#
# try: # In case of interruption
#     for epoch in range(current_epoch, train_config['max_iterations']+1):

#         #--- Quantum discriminator parameter updates ---#
#         for disc_train_step in range(D_STEPS):
#             disc_pass()

#         #--- Quantum generator parameter updates ---#
#         for gen_train_step in range(G_STEPS):
#             gen_pass()

#         #--- Track KL and save best performing generator weights ---#
#         evaluation()

#         # Calculate time
#         cur_time = (time.time() - start_time)
#         times[epoch] = cur_time
#         start_time = time.time()


#         #--- Print progress ---#
#         if train_config['print_progress_iterations'] and (epoch % train_config['print_progress_iterations'] == 0):
#             now_times = sum(times.values())
#             for header, val in zip(TABLE_HEADERS.split('|'),
#                                 (epoch, gen_loss, disc_loss, current_kl, min_eval, now_times - prev_times)):
#                 print(f"{val:.3g} ".rjust(len(header)), end="|")
#             print()

#             prev_times = now_times

#         # In case of interruption
#         if interrupter.kill_now:
#             print("Interrupter: Graceful exit triggered. Breaking loop.")
#             break
            
# #--- Save parameters and optimizer states data ---#
# finally:
#     torch.save({
#         'init_gen_params': params['init_gen_params'],
#         'init_disc_params': params['init_disc_params'],
#         'best_gen_params': best_gen_params,
#         'gen_params': gen_params,
#         'disc_params': disc_params,
#         'optimizer_g_state': optimizer_g.state_dict(),
#         'optimizer_d_state': optimizer_d.state_dict(),
#         'current_epoch': epoch+1,
#         "metrics": {
#             "gloss": gloss,
#             "dloss": dloss,
#             "eval": eval,
#             'times': times,
#         },
#         'random_state': np.random.get_state()
#     }, train_config['training_data_file'])
    
#     eval_data = list(eval.values()) if eval else [0]
#     print("Training complete:", "\n   Data path:", train_config['training_data_file'], "\n   Best eval:", np.min(eval_data), "in epoch", np.argmin(eval_data), "\n   Improvement:", eval_data[0]-np.min(eval_data), "\n   Total time:", sum(times.values()))

In [ ]:
#- Training -#

D_STEPS = train_config['disc_iterations']
G_STEPS = train_config['gen_iterations']
C_STEPS = train_config['save_loss_iterations']

real_distribution_tensor = torch.from_numpy(Statevector(real_circuit).probabilities()).to(device) # Retrieve real data probability distribution 

interrupter = Interrupter()

if train_config['print_progress_iterations']:
    TABLE_HEADERS = "Epoch | Generator cost | Discriminator cost | Eval | Best eval | Time |"
    print(TABLE_HEADERS)

prev_times = 0
start_time = time.time()

#--- Training loop ---#
try: # In case of interruption
    for epoch in range(current_epoch, train_config['max_iterations']+1):

        #--- Quantum discriminator parameter updates ---#
        for disc_train_step in range(D_STEPS):
            gen_params_np = gen_params.detach().cpu().numpy()
            disc_params_np = disc_params.detach().cpu().numpy()

            # Calculate discriminator cost
            if (disc_train_step == D_STEPS - 1) and (epoch % C_STEPS == 0):
                value_dcost_fake = disc_fake_qnn.forward(gen_params_np, disc_params_np)[0,0]
                value_dcost_real = disc_real_qnn.forward([], disc_params_np)[0,0]
                disc_loss = ((value_dcost_real - value_dcost_fake)-2)/4
                dloss[epoch] = disc_loss

            # Caltulate discriminator gradient
            grad_dcost_fake = disc_fake_qnn.backward(gen_params_np, disc_params_np)[1][0,0]
            grad_dcost_real = disc_real_qnn.backward([], disc_params_np)[1][0,0]
            grad_dcost = grad_dcost_real - grad_dcost_fake
            
            # Update discriminator parameters
            optimizer_d.zero_grad()
            disc_params.grad = torch.tensor(grad_dcost, dtype = torch.float32, device=device)
            optimizer_d.step()


        #--- Quantum generator parameter updates ---#
        for gen_train_step in range(G_STEPS):
            gen_params_np = gen_params.detach().cpu().numpy()
            disc_params_np = disc_params.detach().cpu().numpy()

            # Calculate generator cost
            if (gen_train_step == G_STEPS - 1) and (epoch % C_STEPS == 0):
                value_gcost = gen_qnn.forward(disc_params_np, gen_params_np)[0,0]
                gen_loss = (value_gcost-1)/2
                gloss[epoch] = gen_loss

            # Calculate generator gradient
            grad_gcost = gen_qnn.backward(disc_params_np, gen_params_np)[1][0,0]

            # Update generator parameters
            optimizer_g.zero_grad()
            gen_params.grad = torch.tensor(grad_gcost, dtype = torch.float32, device=device)
            optimizer_g.step()


        #--- Track KL and save best performing generator weights ---#
        sv = AerStatevector(generator_circuit.assign_parameters(gen_params_np), device=device)
        gen_dist = torch.tensor(sv.probabilities(), device=device)

        # Performance measurement function: uses Kullback Leibler Divergence to measures the distance between two distributions
        current_kl = evaluate(gen_dist, real_distribution_tensor)
        
        eval[epoch] = current_kl
        if min_eval > current_kl:
            min_eval = current_kl
            best_gen_params = gen_params_np.copy() # New best


        # Calculate time
        cur_time = (time.time() - start_time)
        times[epoch] = cur_time
        start_time = time.time()


        #--- Print progress ---#
        if train_config['print_progress_iterations'] and (epoch % train_config['print_progress_iterations'] == 0):
            now_times = sum(times.values())
            for header, val in zip(TABLE_HEADERS.split('|'),
                                (epoch, gen_loss, disc_loss, current_kl, min_eval, now_times - prev_times)):
                print(f"{val:.3g} ".rjust(len(header)), end="|")
            print()

            prev_times = now_times

        # In case of interruption
        if interrupter.kill_now:
            print("Interrupter: Graceful exit triggered. Breaking loop.")
            break
            
#--- Save parameters and optimizer states data ---#
finally:
    torch.save({
        'init_gen_params': params['init_gen_params'],
        'init_disc_params': params['init_disc_params'],
        'best_gen_params': best_gen_params,
        'gen_params': gen_params,
        'disc_params': disc_params,
        'optimizer_g_state': optimizer_g.state_dict(),
        'optimizer_d_state': optimizer_d.state_dict(),
        'current_epoch': epoch+1,
        "metrics": {
            "gloss": gloss,
            "dloss": dloss,
            "eval": eval,
            'times': times,
        },
        'random_state': np.random.get_state()
    }, train_config['training_data_file'])
    
    eval_data = list(eval.values()) if eval else [0]
    print("Training complete:", "\n   Data path:", train_config['training_data_file'], "\n   Best eval:", np.min(eval_data), "in epoch", np.argmin(eval_data), "\n   Improvement:", eval_data[0]-np.min(eval_data), "\n   Total time:", sum(times.values()))

Epoch | Generator cost | Discriminator cost | Eval | Best eval | Time |
    0 |         -0.126 |             -0.671 | 6.95 |      6.95 | 5.19 |
   10 |         -0.143 |             -0.662 | 6.91 |      6.91 | 12.2 |
   20 |         -0.164 |             -0.651 | 6.37 |      6.37 | 10.3 |
   30 |         -0.192 |              -0.64 | 5.99 |      5.99 | 10.4 |
   40 |         -0.229 |             -0.631 | 5.42 |      5.42 | 10.4 |
   50 |         -0.271 |             -0.625 | 4.71 |      4.71 | 10.8 |
   60 |         -0.315 |             -0.624 | 3.99 |      3.99 | 10.3 |
   70 |         -0.354 |             -0.627 | 3.29 |      3.29 | 10.3 |
   80 |         -0.388 |             -0.632 | 2.89 |      2.89 | 10.6 |
   90 |         -0.422 |             -0.635 | 3.06 |      2.88 | 10.4 |
  100 |          -0.46 |             -0.635 | 3.44 |      2.88 | 10.3 |
  110 |         -0.504 |             -0.629 | 3.91 |      2.88 | 10.3 |
  120 |         -0.551 |              -0.62 | 3.64 |      2.88 |